# 05 · The language of the senses, and the map stretched to fill a rectangle

1. **Senses.** How much does each passage appeal to sight, hearing, smell, taste and touch? Plain text
   mining, no model: one hand-made word list per sense, counted per 1,000 words.
2. **Stretched map.** The passages are placed on the diamond of the four functions, turned 45° into a
   rectangle, and the plane is deformed so that they cover it evenly. Order is kept (a passage more to the
   right than another stays more to the right); distances are not: the scale is relative.
3. **Sensory passages.** A passage is sensory if it is among the 5 % most charged with any of the five senses.

Input (in `DATA_DIR`): `chunks_mapped.csv.gz` (passages with their text, from Zenodo) and `functions_soft.csv.gz`.
Output: `senses.csv.gz` (words of each sense per 1,000 words) and `uniform_xy.csv.gz` (position of each
passage on the rectangle, before and after stretching, and whether it is sensory), used by `06_figure.ipynb`.

## 1. Settings

In [1]:
DATA_DIR = "../data/results"     # folder with the data files (see the README)
TOP_SHARE = 0.05                 # a passage is sensory if it is among the 5 % most charged with a sense
WINDOW = 0.03                    # stretching: "similar x" = within 3 % of the passages on each side
MAX_PER_WORK = 250               # passages per work on the map, so that long works do not dominate

In [2]:
import numpy as np
import pandas as pd

## 2. Word lists

The corpus is modernized and has no accents; the text is lowercased here.
Ambiguous words are left out on purpose: "ver" and "sentir" (too general), "gusto" (pleasure, will),
"tocar" (also "to concern"), "sed" (also "be!"), "vino" (mostly "came"), "carne" (flesh vs. spirit),
and words of sweetness and softness, which belong more to love than to touch.
Example: "el olor de los unguentos" -> 2 smell words ("olor", "unguentos").

In [3]:
SENSES = {
    "vista": r"\b(ojos?|mir(?:a|o|e|an|en|ar|aba|aban|ando|ad|ase)|vista|luz|luces|lumbre|resplandor\w*|resplandec\w*|"
             r"brill\w*|color\w*|hermosur\w*|belleza\w*|semblante\w*|rostro\w*)\b",
    "oido": r"\b(oid[oa]s?|oir|oye\w*|oy[oe]ron|oyo|oia\w*|oigo|oiga\w*|escuch\w*|voz|voces|sonido\w*|son[oa]r\w*|"
            r"ruido\w*|musica\w*|cant(?:ar|o|os|aba|aban|ando|ico|icos)|armonia\w*|clamor\w*|grit\w*|gemid\w*|"
            r"trompeta\w*|campana\w*|silencio\w*)\b",
    "olfato": r"\b(olor\w*|oler|huel[ea]\w*|hed[eo]r\w*|hedi\w*|hediond\w*|fragancia\w*|fragante\w*|aroma\w*|"
              r"perfum\w*|incienso\w*|unguento\w*|balsamo\w*|almizcle|ambar|mirra|nardo\w*)\b",
    "gusto": r"\b(sabor\w*|sabros\w*|sabore\w*|paladar\w*|manjar\w*|miel|hiel|vinagre|amarg\w*|"
             r"com(?:er|e|en|ia|ian|io|ieron|ido|ida|idas|iendo|ed)|comida\w*|beb(?:er|e|en|ia|io|ieron|ido|ida|idas|iendo|ed)|"
             r"hambre\w*|hambrient\w*)\b",
    "tacto": r"\b(toque\w*|tact\w*|palp\w*|frio|fria|frios|frias|calor\w*|caliente\w*|quem\w*|asper\w*|cilicio\w*|"
             r"blandura|llaga\w*|herid\w*|bes[oa]\w*)\b",
}

## 3. Count the words of each sense

In [4]:
chunks = pd.read_csv(f"{DATA_DIR}/chunks_mapped.csv.gz", usecols=["text_id", "chunk_n", "text"])
text = chunks["text"].str.lower()
words = text.str.count(r"\w+").clip(lower=1)
for sense, pattern in SENSES.items():
    chunks[sense] = text.str.count(pattern) / words * 1000

Check: how often each sense appears, and its most frequent words (to spot false friends).

In [5]:
print(chunks[list(SENSES)].describe().loc[["mean", "50%", "max"]].round(2))
print("\nshare of passages with at least one word:")
print((chunks[list(SENSES)] > 0).mean().round(2))
for sense, pattern in SENSES.items():
    found = text.str.findall(pattern).explode().value_counts().head(12)
    print(f"\n{sense}:", ", ".join(f"{w} {n}" for w, n in found.items()))

      vista    oido  olfato  gusto  tacto
mean   3.09    1.61    0.23   1.19   0.82
50%    0.00    0.00    0.00   0.00   0.00
max   84.00  104.84   72.00  72.29  76.00

share of passages with at least one word:
vista     0.42
oido      0.24
olfato    0.03
gusto     0.16
tacto     0.14
dtype: float64



vista: ojos 11657, luz 8368, vista 4522, mira 4209, hermosura 3002, rostro 2914, mirar 2451, mirando 1655, mirad 1527, lumbre 1212, mire 1117, resplandor 1039



oido: voz 3380, oir 2451, voces 2198, oido 1795, silencio 1629, oyo 1281, oidos 1095, oye 1044, oyendo 978, ruido 703, oyentes 626, musica 617



olfato: olor 1173, mirra 321, olores 249, incienso 234, fragancia 210, hedor 166, balsamo 151, unguentos 112, unguento 109, oler 102, hediondo 97, huele 94



gusto: comer 3379, hambre 1795, manjar 1788, comida 1548, beber 1121, come 898, sabor 755, amargura 652, miel 626, manjares 606, comia 385, hiel 384



tacto: llagas 1298, calor 1280, frio 920, heridas 715, herido 711, herida 634, llaga 532, blandura 491, aspereza 487, toque 384, aspero 306, aspera 288


In [6]:
chunks[["text_id", "chunk_n"] + list(SENSES)].to_csv(f"{DATA_DIR}/senses.csv.gz", index=False)
senses = pd.read_csv(f"{DATA_DIR}/senses.csv.gz")

## 4. The passages on the rectangle

Each passage is placed on a diamond, one corner per function, at the average of the corners weighted by its
function percentages (from `03_functions.ipynb`). Then the diamond is turned 45°: (x, y) -> (x − y, x + y).
Example: 100 % interiorise -> (0, 1) on the diamond -> (−1, 1), the top left corner of the rectangle.
Only up to 250 passages per work, evenly spaced along the work.

In [7]:
FUNCTIONS = ["legitimar", "devocion", "normar", "interiorizar"]
CORNER = {"interiorizar": (0, 1), "legitimar": (0, -1), "devocion": (-1, 0), "normar": (1, 0)}

def position(table):
    """Weighted average of the four corners (weights = function percentages)."""
    return sum(np.outer(table[f], CORNER[f]) for f in FUNCTIONS)

soft = pd.read_csv(f"{DATA_DIR}/functions_soft.csv.gz")
passages = pd.read_csv(f"{DATA_DIR}/chunks_mapped.csv.gz", usecols=["text_id", "chunk_n", "title", "author"])
data = soft.merge(passages)
data["work"] = data["author"].str.split(";").str[0].str.split("(").str[0].str.strip() + " | " + data["title"]

capped = []
for work, rows in data.sort_values(["text_id", "chunk_n"]).groupby("work"):
    every = np.linspace(0, len(rows) - 1, min(MAX_PER_WORK, len(rows))).round().astype(int)
    capped.append(rows.iloc[every])
capped = pd.concat(capped)

diamond = position(capped)
u = diamond[:, 0] - diamond[:, 1]
v = diamond[:, 0] + diamond[:, 1]
n = len(u)
print(n, "passages on the map")

23022 passages on the map


## 5. Stretch the map so that passages cover the rectangle evenly

Conditional ranks: x = rank of the passage among all passages; y = rank among the passages with a similar x.
Done in both directions (x first, y first), averaged and ranked again so that neither axis has priority.

In [8]:
def rank(values):
    """Position of each value among all, from 0 to 1. Example: [5, 1, 3] -> [1.0, 0.0, 0.5]."""
    return values.argsort().argsort() / (len(values) - 1)

def conditional_rank(first, second):
    """For each passage, rank of `second` among the passages whose `first` rank is close to its own."""
    first_rank = rank(first)
    order = np.argsort(first_rank)
    sorted_first = first_rank[order]
    sorted_second = second[order]
    result = np.zeros(len(first))
    for i in range(len(first)):
        low = np.searchsorted(sorted_first, first_rank[i] - WINDOW)
        high = np.searchsorted(sorted_first, first_rank[i] + WINDOW)
        neighbors = sorted_second[low:high]
        result[i] = (neighbors < second[i]).mean()
    return result

x_first = (rank(u), conditional_rank(u, v))
y_first = (conditional_rank(v, u), rank(v))
uniform_u = rank((x_first[0] + y_first[0]) / 2) * 2 - 1      # back to -1 … 1
uniform_v = rank((x_first[1] + y_first[1]) / 2) * 2 - 1

Check: is it even now? Share of passages in each of 4 × 4 cells (perfectly even = 6.2 % each).

In [9]:
cells, _, _ = np.histogram2d(uniform_u, uniform_v, bins=4, range=[[-1, 1], [-1, 1]])
print((cells.T[::-1] / n * 100).round(1))
print("order kept along each axis (Spearman r):",
      round(np.corrcoef(rank(u), rank(uniform_u))[0, 1], 3), round(np.corrcoef(rank(v), rank(uniform_v))[0, 1], 3))

[[6.7 6.8 5.9 5.5]
 [6.6 6.6 5.9 5.9]
 [5.7 6.  5.3 8. ]
 [6.  5.6 7.8 5.6]]
order kept along each axis (Spearman r): 0.969 0.965


## 6. Sensory passages, and save

A passage is sensory if it is in the top 5 % of at least one sense (and has at least one word of it).

In [10]:
positions = capped[["text_id", "chunk_n"]].copy()
positions["u"], positions["v"], positions["uniform_u"], positions["uniform_v"] = u, v, uniform_u, uniform_v

limits = {s: max(senses[s].quantile(1 - TOP_SHARE), senses.loc[senses[s] > 0, s].min()) for s in SENSES}
scores = positions.merge(senses, how="left").fillna(0)
sensory = np.zeros(n, dtype=bool)
for s in SENSES:
    sensory |= (scores[s] >= limits[s]).to_numpy()
print("thresholds (words per 1,000):", {s: round(float(x), 1) for s, x in limits.items()})
print(f"sensory passages on the map: {sensory.sum()} of {n} ({sensory.mean():.0%})")

positions["sensory"] = sensory
positions.to_csv(f"{DATA_DIR}/uniform_xy.csv.gz", index=False)

thresholds (words per 1,000): {'vista': 12.0, 'oido': 8.0, 'olfato': 3.1, 'gusto': 8.0, 'tacto': 4.0}
sensory passages on the map: 5453 of 23022 (24%)
